# X-CLIP base/32 — DIMER zero-shot video classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/xclip-video-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/xclip-video-classification-pipeline/blob/main/tutorials/xclip_video_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-microsoft%2Fxclip--base--patch32-ffcc4d?style=flat)](https://huggingface.co/microsoft/xclip-base-patch32) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FVideoX-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/VideoX/tree/master/X-CLIP) [![arXiv](https://img.shields.io/badge/arXiv-2208.02816-b31b1b.svg)](https://arxiv.org/abs/2208.02816)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot video classification — one 8-frame clip plus 2–32 free-text class names → a ranking of those names with a softmax over them — using the pinned `microsoft/xclip-base-patch32` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/xclip_video_classification_pipeline/pipeline.py` at revision `758d2e3fc5e0`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `a2e27a78a2b5d802e894b8a1ef14f3a8ce490963` (~790 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the X-CLIP model (a CLIP ViT-B/32 frame encoder with cross-frame attention, a one-layer multi-frame integration transformer that fuses the 8 frame embeddings into one video embedding, a CLIP text encoder, and a video-specific prompt generator; about 197M parameters, trained fully supervised on Kinetics-400) embeds the clip and each class name and scores every pair; the carried module softmaxes the scores over the names you supplied. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (exactly 8 frames of one size within the side ceilings, 2–32 distinct class names up to 64 characters), a fixed output contract (probability, logit and rank per name, the top-1 name), and the `sample_frames`, `frames_from_animation`, `validate_inputs` and `evaluation_report` helpers. The default sample is five 8-frame clips of moving shapes drawn in code with their intended class names, so `top1_accuracy` against the chance baseline is demonstration (plumbing) evidence for cartoons, not a Kinetics benchmark — and the model gets one of the five right, which the notebook keeps and explains: it was trained on human-action video and does not read the motion of flat drawn shapes.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw synthetic 8-frame clips (or upload your own animated GIF/WebP and subsample it) and validate them into an input manifest, name the candidate classes, run the supported task, read the ranking correctly (a softmax over your own label set, not a calibrated probability), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `top1_accuracy` and a chance baseline only when the true classes are known and `not-measurable` otherwise, and export the rankings, a contact sheet and provenance.

**This notebook does not demonstrate:** Video decoding from container formats (no `.mp4`/`.avi` reader is pinned; the notebook accepts frames or an animated image Pillow can open), temporal localisation or per-frame labels (one ranking per clip), open-set or abstaining classification (the softmax always picks one of your names), video–text retrieval over a corpus, batch throughput, evaluation on Kinetics-400 or UCF101 (not bundled; only drawn clips are scored here), and any training. The model was trained on human-action video at 8 frames; drawn shapes, static scenes, screen recordings and non-English class names are outside what this notebook measures, and a confident ranking carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 5.2 s to load and about 0.1 s per 8-frame clip with five names in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 786 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python, NumPy and PIL; what a contrastive video–text model scores; why a softmax over a label set you chose is a ranking and not a probability; what top-1 accuracy against a chance baseline does and does not show on five clips.
- **Data:** the default sample is five deterministic 8-frame clips at 320×240 drawn in code with Pillow (a ball rolling right, a ball bouncing, a square growing, a sun setting with a darkening sky, a ball standing still; no text rendering, so their digests are stable across Pillow builds) with the five class names that describe them, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one animated GIF/WebP/APNG that Pillow can open (at least 8 frames; subsampled uniformly to 8), any colour mode, sides between 16 and 4096 px, plus your own class names typed into the form field; the true class is unknown for uploads, so their report is `not-measurable`. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `microsoft/xclip-base-patch32` snapshot (~790 MB in total) at revision `a2e27a78a2b5…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'xclip-video-classification-pipeline',
    'repository_revision': '758d2e3fc5e01edd851eda97754c7cf6ae8daf8e',
    'embedded_module': 'src/xclip_video_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/xclip_video_classification_pipeline/pipeline.py'],
    'module_sha256': '6990ffed45fa75e8fcdf2f3ca65dd8d222209cfde9e9fc2cf4e3ca973ff80567',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/xclip_video_classification_pipeline/` @ `758d2e3fc5e0`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/xclip_video_classification_pipeline/pipeline.py`

In [ ]:
"""Zero-shot video classification with the pinned ``microsoft/xclip-base-patch32`` checkpoint (X-CLIP).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the X-CLIP architecture comes from the pinned ``transformers`` release, the
weights are SafeTensors, and no model-repository code is executed. A clip is a sequence of exactly
NUM_FRAMES PIL frames; the caller names the candidate classes as free text and receives a softmax over
those names — a relative ranking, not a calibrated probability.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image, ImageSequence

MODEL_ID = "microsoft/xclip-base-patch32"
MODEL_REVISION = "a2e27a78a2b5d802e894b8a1ef14f3a8ce490963"
MODEL_LICENSE = "mit"
MODEL_KEY = "xclip-base-patch32"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The checkpoint was trained on 8 frames per clip (config.json vision_config.num_frames); the temporal
# modules expect exactly that many, so a clip is exactly NUM_FRAMES frames and longer sequences are
# subsampled uniformly by sample_frames.
NUM_FRAMES = 8
# Frame preprocessing: shorter side resized to 224, centre crop 224x224, ImageNet mean/std
# (preprocessor_config.json), 32x32 patches -> 49 tokens per frame.
FRAME_SIZE = 224
# Input ceilings. Each label is one CLIP text query (77-token context); the label set is the caller's
# closed vocabulary for this request.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MIN_LABELS = 2
MAX_LABELS = 32
MAX_LABEL_CHARS = 64
MAX_TEXT_TOKENS = 77


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def format_labels(labels: Sequence[str]) -> list[str]:
    """Validate the candidate class names and normalise them: stripped, whitespace-collapsed,
    lower-cased, trailing full stop removed, distinct. Names are passed to the CLIP text tower as-is
    otherwise (the upstream zero-shot recipe uses plain class names such as "playing soccer")."""
    if isinstance(labels, str) or not isinstance(labels, Sequence):
        raise TypeError("labels must be a list of class names, not a single string")
    if not MIN_LABELS <= len(labels) <= MAX_LABELS:
        raise ValueError(
            f"label count {len(labels)} outside MIN_LABELS {MIN_LABELS}..MAX_LABELS {MAX_LABELS}"
        )
    cleaned: list[str] = []
    for name in labels:
        if not isinstance(name, str):
            raise TypeError(f"label must be str, got {type(name).__name__}")
        text = " ".join(name.split()).strip().rstrip(".").strip().lower()
        if not text:
            raise ValueError("labels must not be empty")
        if len(text) > MAX_LABEL_CHARS:
            raise ValueError(
                f"label {text[:12]!r}... is {len(text)} chars > MAX_LABEL_CHARS {MAX_LABEL_CHARS}"
            )
        cleaned.append(text)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("labels must be distinct after normalisation")
    return cleaned


def validate_frame(frame: Any) -> Image.Image:
    if not isinstance(frame, Image.Image):
        raise TypeError(f"frame must be a PIL.Image.Image, got {type(frame).__name__}")
    width, height = frame.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"frame side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"frame side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return frame.convert("RGB")


def validate_clip(frames: Any) -> list[Image.Image]:
    """Exactly NUM_FRAMES PIL frames of one common size, each within the side ceilings."""
    if isinstance(frames, Image.Image) or not isinstance(frames, Sequence):
        raise TypeError("frames must be a sequence of PIL.Image.Image, not a single image")
    if len(frames) != NUM_FRAMES:
        raise ValueError(
            f"a clip is exactly NUM_FRAMES={NUM_FRAMES} frames, got {len(frames)}; "
            "use sample_frames to subsample a longer sequence"
        )
    rgb = [validate_frame(frame) for frame in frames]
    if len({frame.size for frame in rgb}) != 1:
        raise ValueError("all frames of a clip must have the same size")
    return rgb


def sample_frames(frames: Sequence[Image.Image], n: int = NUM_FRAMES) -> list[Image.Image]:
    """Pick ``n`` frames at evenly spaced indices (first and last included) from a longer sequence."""
    if isinstance(frames, Image.Image) or not isinstance(frames, Sequence):
        raise TypeError("frames must be a sequence of PIL.Image.Image")
    if len(frames) < n:
        raise ValueError(f"need at least {n} frames to sample {n}, got {len(frames)}")
    indices = np.linspace(0, len(frames) - 1, num=n).round().astype(int)
    return [frames[int(index)] for index in indices]


def frames_from_animation(image: Image.Image) -> list[Image.Image]:
    """Decode every frame of an animated image (GIF, WebP, APNG) that Pillow can open into RGB copies."""
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    frames = [frame.convert("RGB") for frame in ImageSequence.Iterator(image)]
    if not frames:
        raise ValueError("the image holds no frames")
    return frames


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        f"one clip of exactly NUM_FRAMES={NUM_FRAMES} PIL.Image.Image frames of one size (any mode, "
        "converted to RGB) plus MIN_LABELS..MAX_LABELS free-text class names"
    ),
    "frame_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "frames_per_clip": NUM_FRAMES,
    "labels": [MIN_LABELS, MAX_LABELS],
    "label_chars": [1, MAX_LABEL_CHARS],
    "label_tokens": [1, MAX_TEXT_TOKENS],
    "preprocessing": (
        f"each frame: shorter side resized to {FRAME_SIZE}, centre crop {FRAME_SIZE}x{FRAME_SIZE}, ImageNet "
        "mean/std, 32x32 patches; labels normalised into one CLIP text query each (format_labels); the "
        "video embedding (frame features fused by the multi-frame integration transformer) is scored "
        "against each label embedding and the scores are softmaxed over the supplied labels"
    ),
    "output": (
        "one probability per supplied label (softmax over the label set: a relative ranking that sums to "
        "1 and is not calibrated), the raw logits, and the top-1 label"
    ),
}


def _check_inputs(frames: Any, labels: Any) -> tuple[list[Image.Image], list[str]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``classify`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    return validate_clip(frames), format_labels(labels)


def validate_inputs(
    clips: Sequence[Sequence[Image.Image]],
    labels: Sequence[str],
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every clip is checked exactly as ``classify`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(clips, Image.Image) or not isinstance(clips, Sequence) or not clips:
        raise TypeError("clips must be a non-empty sequence of frame sequences")
    if clips and isinstance(clips[0], Image.Image):
        raise TypeError("clips must be a sequence of clips (each a sequence of frames), not one clip")
    if names is not None and len(names) != len(clips):
        raise ValueError(f"names has {len(names)} entries for {len(clips)} clips")
    checked_labels = format_labels(labels)
    observed = []
    for index, clip in enumerate(clips):
        rgb, _ = _check_inputs(clip, labels)
        observed.append(
            {
                "id": names[index] if names else f"clip-{index}",
                "n_frames": len(rgb),
                "frame_mode": clip[0].mode,
                "frame_size": list(rgb[0].size),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": observed,
        "labels": checked_labels,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    correct_labels: Sequence[str] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``correct_labels`` (one class name per result, in order; normalised like the labels) the report
    carries ``top1_accuracy`` over the clips, the chance baseline (mean of 1/n_labels) and one
    per-clip entry, verdict ``sample-sanity``; without them it is ``not-measurable`` and says what
    labelled data would make the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one classification result")
    base = {
        "task": "zero-shot video classification over a caller-supplied label set (top-1 over the labels)",
        "score_semantics": (
            "probabilities are a softmax over the supplied labels only: a relative ranking that sums to 1, "
            "not a calibrated probability, and a label set without the true class still yields a confident "
            "top-1"
        ),
        "sample_kind": sample_kind,
        "n_clips": len(results),
        "n_labels": [len(result["labels"]) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if correct_labels is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no correct labels were supplied for the classified clips",
            "needs": (
                "labelled clips from the deployment domain with a class vocabulary matching the labels "
                "(Kinetics-style annotations) scored with top-1/top-5 accuracy over many clips; no such "
                "labelled set ships with this repository"
            ),
        }
    if len(correct_labels) != len(results):
        raise ValueError(f"correct_labels has {len(correct_labels)} entries for {len(results)} results")
    per_clip = []
    for result, correct in zip(results, correct_labels, strict=True):
        key = format_labels([correct, "__second__"])[0]
        if key not in result["labels"]:
            raise ValueError(f"correct label {correct!r} is not among the result's labels")
        per_clip.append(
            {
                "clip": result.get("clip"),
                "top1": result["top1"],
                "top1_probability": result["predictions"][0]["probability"],
                "correct_label": key,
                "correct": result["top1"] == key,
                "rank_of_correct": next(
                    index for index, entry in enumerate(result["predictions"]) if entry["label"] == key
                )
                + 1,
            }
        )
    chance = sum(1.0 / n for n in base["n_labels"]) / len(results)
    return {
        **base,
        "metrics": [
            {
                "id": "top1_accuracy",
                "value": sum(entry["correct"] for entry in per_clip) / len(per_clip),
                "estimation": f"{len(per_clip)} clip(s), no dispersion estimate",
            }
        ],
        "baselines": [{"id": "chance", "value": chance, "note": "mean of 1/n_labels over the clips"}],
        "per_clip": per_clip,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_clip)} clip(s) with caller-known classes; plumbing evidence, not a video "
            "classification benchmark"
        ),
        "needs": (
            "labelled clips from the deployment domain with a matching class vocabulary for any "
            "top-1/top-5 accuracy claim; Kinetics-400 and UCF101 are not bundled"
        ),
    }


@dataclass
class XClipVideoClassificationPipeline:
    """Zero-shot video classification over the pinned X-CLIP base/32 checkpoint."""

    _runner: Callable[[list[Image.Image], list[str]], np.ndarray]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> XClipVideoClassificationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import XCLIPModel, XCLIPProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = XCLIPProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = XCLIPModel.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float32, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(frames: list[Image.Image], labels: list[str]) -> np.ndarray:
            # The frame list is handed to the snapshot's VideoMAE-style image processor directly: in
            # the pinned transformers release XCLIPProcessor's `videos=` keyword yields no
            # pixel_values (only `images=` does), and calling the two sub-processors makes the
            # contract explicit. One clip -> pixel_values (1, NUM_FRAMES, 3, 224, 224).
            pixel_values = processor.image_processor([frames], return_tensors="pt")["pixel_values"]
            text = processor.tokenizer(labels, padding=True, return_tensors="pt")
            with torch.inference_mode():
                outputs = model(
                    pixel_values=pixel_values.to(resolved_device),
                    input_ids=text["input_ids"].to(resolved_device),
                    attention_mask=text["attention_mask"].to(resolved_device),
                )
            return outputs.logits_per_video[0].float().cpu().numpy()

        return cls(runner, resolved_device)

    def classify(self, frames: Sequence[Image.Image], labels: Sequence[str]) -> dict[str, Any]:
        """Rank ``labels`` for one clip of exactly NUM_FRAMES frames; probabilities are a softmax over them.

        The softmax is a relative ranking over the supplied labels, not a calibrated probability.
        """
        rgb, names = _check_inputs(frames, labels)
        logits = np.asarray(self._runner(rgb, names), dtype=np.float64).reshape(-1)
        if logits.shape != (len(names),) or not np.all(np.isfinite(logits)):
            raise RuntimeError(f"backend returned logits of shape {logits.shape} for {len(names)} labels")
        shifted = np.exp(logits - logits.max())
        probs = shifted / shifted.sum()
        order = np.argsort(-probs)
        predictions = [
            {"label": names[index], "probability": float(probs[index]), "logit": float(logits[index])}
            for index in order
        ]
        return {
            "predictions": predictions,
            "top1": predictions[0]["label"],
            "labels": names,
            "n_frames": len(rgb),
            "frame_size": list(rgb[0].size),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `9`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a2e27a78a2b5…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `XClipVideoClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "xclip-base-patch32",
  "modelId": "microsoft/xclip-base-patch32",
  "revision": "a2e27a78a2b5d802e894b8a1ef14f3a8ce490963",
  "files": [
    {
      "path": "README.md",
      "bytes": 2728,
      "sha256": "5b4c29da30c14c41dadc2320cedb68d7bc30147aa171c04ce8d3d7b8acf8da91"
    },
    {
      "path": "config.json",
      "bytes": 4718,
      "sha256": "13bb919d1ef16f3b80b03cdf5c575d688dfafd21d12566849cc557b00e058ce3"
    },
    {
      "path": "merges.txt",
      "bytes": 524657,
      "sha256": "f526393189112391ce6f9795d4695f704121ce452c3aad1f5335cc41337eba85"
    },
    {
      "path": "model.safetensors",
      "bytes": 786414772,
      "sha256": "abf286e8cdd0612761c3e42d3a55eca998382dfa67a04a0f3fdcdfa4f150cdbb"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 309,
      "sha256": "c14b2b5c8f26a754df62235ba79d1ca63cfdd9b3de76ee688e4a30ea1e5c6986"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 472,
      "sha256": "c4864a9376a8401918425bed71fc14fc0e81f9b59ec45c1cf96cccb2df508eac"
    },
    {
      "path": "tokenizer.json",
      "bytes": 2224041,
      "sha256": "a75dc79c6ec004a7e2d346c20e0af8d29aa2b251ea356964718aef8b8f052e80"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 965,
      "sha256": "7e28810d55aafb02f6b216f05fa1208a4aac62abf794796288a70314eed5ddf3"
    },
    {
      "path": "vocab.json",
      "bytes": 862328,
      "sha256": "5047b556ce86ccaf6aa22b3ffccfc52d391ea4accdab9c2f2407da5b742d4363"
    }
  ],
  "totalBytes": 790034990
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = XClipVideoClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic clips or optional BYOD

The default sample is **synthetic** and carries its own references: five 8-frame clips at 320×240 — a red ball rolling left to right along green ground, a ball bouncing in place, a blue square growing, a sun sinking while the sky darkens, and a ball standing still — are drawn with Pillow, the same clips the repository's smoke run used, and the five class names describing them are the label set. The intended class of each clip is the reference for the `top1_accuracy` sanity check later. They are not a labelled dataset, so nothing here is a Kinetics measurement — and the smoke run ranked `a ball standing still` first for every clip except the sunset's runner-up, scoring 1/5 (chance 1/5): the model does not read the motion of flat drawn shapes, which the notebook keeps as a recorded finding rather than tuning the drawings until they pass. Each clip's digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one animated image and type your class names — its true class is unknown, so the evaluation report will be `not-measurable`.

The label set is a **caller-owned request parameter**: the softmax ranks only the names you supply, so a set that omits the true class still yields a confident top-1. Nothing is validated in this cell — the next section hands the clips and the names to the pipeline's own validation stage, which is the only checker. Look for one dictionary per clip naming the sample kind, frame count, size and digest, plus the label set.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
byod_labels = 'playing soccer\ndancing\ncooking'  # @param {type:"string"}


def synthetic_clip(kind, n=8, size=(320, 240)):
    """An 8-frame cartoon clip drawn with Pillow (no text): sky, green ground and one moving element."""
    frames = []
    for index in range(n):
        t = index / (n - 1)
        sky = (135, 206, 235)
        if kind == 'the sun setting':
            sky = (int(135 * (1 - t) + 30 * t), int(206 * (1 - t) + 40 * t), int(235 * (1 - t) + 80 * t))
        frame = Image.new('RGB', size, sky)
        d = ImageDraw.Draw(frame)
        d.rectangle([0, 170, 320, 240], fill=(60, 179, 75))  # ground
        if kind == 'a ball rolling to the right':
            x = 30 + t * 230
            d.ellipse([x, 130, x + 40, 170], fill=(220, 40, 40))
        elif kind == 'a ball bouncing up and down':
            y = 130 - abs(np.sin(t * np.pi * 2)) * 100
            d.ellipse([140, y, 180, y + 40], fill=(220, 40, 40))
        elif kind == 'a square growing larger':
            s = 10 + t * 90
            d.rectangle([160 - s / 2, 120 - s / 2, 160 + s / 2, 120 + s / 2], fill=(40, 70, 200))
        elif kind == 'the sun setting':
            y = 30 + t * 140
            d.ellipse([240, y, 290, y + 50], fill=(255, 215, 0))
        elif kind == 'a ball standing still':
            d.ellipse([140, 130, 180, 170], fill=(220, 40, 40))
        frames.append(frame)
    return frames


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    clip_name = next(iter(uploaded))
    animation = Image.open(io.BytesIO(uploaded[clip_name]))
    clips = [sample_frames(frames_from_animation(animation))]
    clip_names = [clip_name]
    labels = [line.strip() for line in byod_labels.splitlines() if line.strip()]
    correct_labels = None
    sample_kind = 'BYOD'
else:
    # Deterministic drawings: no randomness and no text rendering, so no seed is needed and the digests are stable.
    labels = ['a ball rolling to the right', 'a ball bouncing up and down', 'a square growing larger', 'the sun setting', 'a ball standing still']
    clips = [synthetic_clip(kind) for kind in labels]
    clip_names = [f"synthetic_{kind.replace(' ', '_')}_8x320x240" for kind in labels]
    correct_labels = list(labels)  # clip i was drawn to depict labels[i]
    sample_kind = 'synthetic'

digests = {name: hashlib.sha256(b''.join(np.asarray(frame.convert('RGB')).tobytes() for frame in clip)).hexdigest() for name, clip in zip(clip_names, clips)}
for name, clip in zip(clip_names, clips):
    print({'sample_kind': sample_kind, 'name': name, 'n_frames': len(clip), 'frame_size': clip[0].size, 'rgb_sha256': digests[name]})
print({'labels': labels, 'has_correct_labels': correct_labels is not None})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `classify` applies — each clip exactly `NUM_FRAMES` PIL frames of one size with sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, and `MIN_LABELS`..`MAX_LABELS` distinct non-empty class names of at most `MAX_LABEL_CHARS` characters (normalised by `format_labels`) — and returns an **input manifest** naming the schema (including the resize/centre-crop preprocessing and the softmax rule), each clip's observed frame count, mode and size, the normalised label set and the verdict. The manifest is written to `outputs/xclip_video_classification_input_manifest.json`. To show what rejection looks like, the cell also validates a 3-frame clip and records the pipeline's own error message as a finding. Inside the pipeline each frame is converted to RGB, resized on its shorter side to 224 and centre-cropped; nothing else is dropped or altered. The pipeline cannot tell whether a clip shows an action or whether your label set contains its true class: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'NUM_FRAMES': NUM_FRAMES, 'FRAME_SIZE': FRAME_SIZE, 'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MIN_LABELS': MIN_LABELS, 'MAX_LABELS': MAX_LABELS, 'MAX_LABEL_CHARS': MAX_LABEL_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS}})
input_manifest = validate_inputs(clips, labels, names=clip_names)
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs([clips[0][:3]], labels)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'three-frame-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/xclip_video_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Classify the clips and read the output correctly

`classify` returns, per clip, `predictions` ordered by descending probability (each with the normalised `label`, its `probability` and raw `logit`), the `top1` name, the normalised `labels`, the frame count and size, and the model identity. **The probabilities are a softmax over your label set**: they sum to 1 and rank the names you supplied, they are not calibrated, and a label set without the true class still yields a confident winner. The forward pass is deterministic on a fixed device and dtype; CUDA kernels can shift logits slightly, so GPU and CPU rankings need not agree on close pairs. Each clip costs one 8-frame encoding plus one text encoding per name (about 0.1 s on the reference CPU). As recorded in the model card, the repository's CPU smoke on these same clips ranked `a ball standing still` first for all five (0.43–0.69), putting the correct name at rank 3–4 for the four moving clips, and ranked the same name first for eight blank white frames (0.70): the model always produces a ranking, whether or not the clip shows an action.

In [ ]:
import time

results, seconds = [], []
for name, clip in zip(clip_names, clips):
    t0 = time.time()
    result = pipe.classify(clip, labels)
    result['clip'] = name
    results.append(result)
    seconds.append(round(time.time() - t0, 2))
print({'device': pipe.device, 'seconds_per_clip': seconds, 'n_labels': len(results[0]['labels'])})
for result in results:
    ranking = ', '.join(f"{entry['label']}={entry['probability']:.2f}" for entry in result['predictions'])
    print(f"{result['clip']}\n   top-1: {result['top1']!r}  |  {ranking}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No accuracy is reported by default: video-classification accuracy needs labelled clips from the deployment domain with a matching class vocabulary, and this repository ships none (Kinetics-400 and UCF101 are not bundled). When the true class of each clip is supplied the report carries `top1_accuracy` over the clips, a `chance` baseline (the mean of 1/labels over the clips) and one entry per clip with the rank of the correct name, with the verdict `sample-sanity`. On the synthetic path those classes are intentions **you drew yourself**, so the score proves only that the input contract, preprocessing, forward pass and softmax round-trip — and the four recorded misses show what chance-level ranking looks like in the report. On BYOD the true class is unknown, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/xclip_video_classification_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, correct_labels, sample_kind=sample_kind)
with open('outputs/xclip_video_classification_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'per_clip', 'baselines')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:15} {metric['value']:.3f}  ({metric['estimation']})")
for baseline in report['baselines']:
    print(f"{baseline['id']:15} {baseline['value']:.3f}  ({baseline['note']})")
for entry in report.get('per_clip', []):
    print(f"  {'OK  ' if entry['correct'] else 'MISS'}  {entry['clip']} -> {entry['top1']!r} {entry['top1_probability']:.2f} (correct {entry['correct_label']!r} at rank {entry['rank_of_correct']})")
if report['verdict'] == 'not-measurable':
    print('The true class of this clip is not known to the notebook, so nothing is scored; judge the ranking yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves every ranking (clip, ordered predictions with probability and logit, top-1, the label set), the evaluation report, the input manifest, the sample identities and digests, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The rankings are also written as CSV with explicit `clip`, `rank`, `label`, `probability`, `logit` columns so ordering survives downstream use, and a contact-sheet PNG shows the 8 frames of every clip with its top-1 name for visual inspection (a supplement to, not a replacement for, the machine-readable files). No credentials are recorded.

In [ ]:
import csv

from PIL import ImageFont

thumb_w, thumb_h, row_h = 120, 90, 90 + 28
sheet = Image.new('RGB', (thumb_w * NUM_FRAMES, row_h * len(clips)), 'white')
draw = ImageDraw.Draw(sheet)
panel_font = ImageFont.load_default(size=14)
for row, (clip, result) in enumerate(zip(clips, results)):
    for col, frame in enumerate(clip):
        thumb = frame.convert('RGB').copy()
        thumb.thumbnail((thumb_w, thumb_h))
        sheet.paste(thumb, (col * thumb_w, row * row_h))
    draw.text((6, row * row_h + thumb_h + 6), f"{result['clip']}  ->  {result['top1']} ({result['predictions'][0]['probability']:.2f})", fill=(40, 90, 220), font=panel_font)
sheet.save('outputs/xclip_video_classification_contact_sheet.png')
payload = {
    'predictions': results,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'names': clip_names, 'rgb_sha256': digests, 'labels': labels, 'correct_labels': correct_labels},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/xclip_video_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/xclip_video_classification_rankings.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['clip', 'rank', 'label', 'probability', 'logit'])
    for result in results:
        for rank, entry in enumerate(result['predictions'], start=1):
            writer.writerow([result['clip'], rank, entry['label'], f"{entry['probability']:.6f}", f"{entry['logit']:.4f}"])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The rankings are softmaxes over the class names you supplied; nothing in the output says whether any of those names fits the clip, the probabilities are relative and uncalibrated, and the model ranks every clip — including eight blank frames — with equal confidence. On the drawn clips the `top1_accuracy` in the evaluation report compares the top-1 name with the class you drew each clip to depict and the verdict is `sample-sanity`, which proves only that the input contract, preprocessing, forward pass and softmax work (the repository's smoke run scored 1/5 against a chance baseline of 0.2, ranking `a ball standing still` first for every clip: a Kinetics-trained model does not read the motion of flat cartoon shapes); it says nothing about real footage of people, sports or everyday actions, clips longer or shorter than the 8 sampled frames, fine-grained action distinctions, or non-English names, and a BYOD result is a single-clip observation with the verdict `not-measurable`. **The label set is part of the request**: the same clip ranked `juggling balls` at 0.98 under Kinetics-style names in the smoke run, so choose names that cover what the clip could show and treat a confident top-1 for a clip that shows none of them as the expected failure mode, not an exception. The pipeline provides no video decoding, no temporal localisation, no abstention, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** replace the label set with Kinetics-style action names (`playing soccer`, `juggling balls`, `sunset`) and watch the ranking change entirely; drop `a ball standing still` from the set and see which name inherits the clips; enable `USE_BYOD` with an animated GIF of a real action you know, then pass its class as `correct_labels` to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/xclip-video-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/xclip-video-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/xclip-video-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/microsoft/xclip-base-patch32
- Upstream code: https://github.com/microsoft/VideoX/tree/master/X-CLIP
- Expanding Language-Image Pretrained Models for General Video Recognition (Ni et al., 2022): https://arxiv.org/abs/2208.02816
- The Kinetics Human Action Video Dataset (Kay et al., 2017): https://arxiv.org/abs/1705.06950